In [2]:
import matsim
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from glob import glob

network_with_height = matsim.read_network("scenarios/Zurich_10pct/zurich_10pct_network.xml")
# This one really does not work:
# network_with_height = matsim.read_network("scenarios/Zurich_10pct/zurich_10pct_network_with_bad_z.xml")


print(network_with_height)
print(network_with_height.network_attrs)


nodes_gdf = gpd.GeoDataFrame(
    network_with_height.nodes[['node_id', 'z']],
    geometry=[Point(x, y) for x, y in zip(network_with_height.nodes.x, network_with_height.nodes.y)],
    crs='EPSG:2056'
)

print(nodes_gdf.head(10))


Network: 55214 nodes, 122687 links, Atlantis
{'coordinateReferenceSystem': 'Atlantis'}
       node_id       z                         geometry
0   1000138721  392.54   POINT (2689122.07 1281700.625)
1  10001518780  391.30  POINT (2672602.677 1250951.352)
2    100016591  443.69  POINT (2680227.888 1254504.487)
3    100017223  439.07  POINT (2679129.241 1254770.953)
4    100018715  442.73  POINT (2678533.155 1254806.624)
5    100018718  441.31  POINT (2678501.564 1254958.819)
6    100018989  440.81  POINT (2678401.262 1254425.627)
7   1000193915  494.35  POINT (2689660.867 1281128.813)
8   1000198181  496.97  POINT (2690176.857 1280750.867)
9   1000204287  469.38  POINT (2690218.997 1280644.798)


In [3]:
# turn to EPSG:4326 
nodes_gdf_google_maps = nodes_gdf.to_crs(epsg=4326)

# Add a normal field, which is the y latitude
nodes_gdf['lat'] = nodes_gdf_google_maps.geometry.y
nodes_gdf['lon'] = nodes_gdf_google_maps.geometry.x
print(nodes_gdf.head(10))

       node_id       z                         geometry        lat       lon
0   1000138721  392.54   POINT (2689122.07 1281700.625)  47.679869  8.625495
1  10001518780  391.30  POINT (2672602.677 1250951.352)  47.405378  8.400505
2    100016591  443.69  POINT (2680227.888 1254504.487)  47.436448  8.502154
3    100017223  439.07  POINT (2679129.241 1254770.953)  47.438977  8.487639
4    100018715  442.73  POINT (2678533.155 1254806.624)  47.439369  8.479744
5    100018718  441.31  POINT (2678501.564 1254958.819)  47.440742  8.479352
6    100018989  440.81  POINT (2678401.262 1254425.627)  47.435959  8.477929
7   1000193915  494.35  POINT (2689660.867 1281128.813)  47.674654  8.632553
8   1000198181  496.97  POINT (2690176.857 1280750.867)  47.671185  8.639346
9   1000204287  469.38  POINT (2690218.997 1280644.798)  47.670225  8.639886


In [4]:


# Read XYZ file directly. You can get it at: https://www.swisstopo.admin.ch/en/height-model-swissaltiregio
root_dir = "data/heights"


dfs = []


for csv_path in glob(f"{root_dir}/**/*.xyz/*.xyz", recursive=True):
    if '2675_1254' not in csv_path:
        continue
    print(f"Reading {csv_path}...")
    df = pd.read_csv(csv_path, sep=' ', header=1, names=['X', 'Y', 'Z'])
    # df = df.sample(n=1000, random_state=42)
    dfs.append(df)
    # do something with csv_path


big_df = pd.concat(dfs, ignore_index=True)



print(big_df.head(10))

# Create GeoDataFrame from the points
geometry = [Point(xy) for xy in zip(big_df['X'], big_df['Y'])]
gdf_heights = gpd.GeoDataFrame(big_df, geometry=geometry, crs='EPSG:2056')  # Coordinate system: LV95 (EPSG 2056)

print(f"Loaded {len(gdf_heights)} points")
print(gdf_heights.head())
print(f"Z range: {big_df['Z'].min():.2f} to {big_df['Z'].max():.2f} meters")

Reading data/heights/swissaltiregio_2675-1254_2056_5728.xyz/2675_1254.xyz...
           X        Y       Z
0  2675015.0  1263995  510.11
1  2675025.0  1263995  509.16
2  2675035.0  1263995  508.16
3  2675045.0  1263995  507.12
4  2675055.0  1263995  505.94
5  2675065.0  1263995  504.51
6  2675075.0  1263995  503.11
7  2675085.0  1263995  502.14
8  2675095.0  1263995  501.10
9  2675105.0  1263995  500.23
Loaded 999999 points
           X        Y       Z                 geometry
0  2675015.0  1263995  510.11  POINT (2675015 1263995)
1  2675025.0  1263995  509.16  POINT (2675025 1263995)
2  2675035.0  1263995  508.16  POINT (2675035 1263995)
3  2675045.0  1263995  507.12  POINT (2675045 1263995)
4  2675055.0  1263995  505.94  POINT (2675055 1263995)
Z range: 396.57 to 645.87 meters


In [5]:
# reduce number of points for plotting to a random sample of 10000
# gdf_heights = gdf_heights.sample(n=1000, random_state=42)

In [6]:
# remove all entries from nodes_gdf exceot 100016591
single_entry = nodes_gdf[nodes_gdf['node_id'] == '100016591']

print(single_entry)

result = gpd.sjoin_nearest(single_entry, gdf_heights, how='left', distance_col='dist_m')
    
result = result.drop(columns=['index_right'], axis=1)

print(result.head(10))
# print length
print(len(result))

     node_id       z                         geometry        lat       lon
2  100016591  443.69  POINT (2680227.888 1254504.487)  47.436448  8.502154
     node_id       z                         geometry        lat       lon  \
2  100016591  443.69  POINT (2680227.888 1254504.487)  47.436448  8.502154   

           X        Y       Z    dist_m  
2  2680225.0  1254505  443.69  2.933302  
1
